# Zero-to-One Launch Syndicate Sandbox
Test the multi-agent pipeline and strict formatting logic before deploying to Streamlit.

In [ ]:
!pip install groq

**Note:** Ensure `GROQ_API_KEY` is configured in Colab Secrets (🔑 icon).

In [ ]:
import os
import json
from google.colab import userdata
from groq import Groq

client = Groq(api_key=userdata.get('GROQ_API_KEY'))

def call_agent(persona, user_input, json_mode=False):
    res = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": persona},
            {"role": "user", "content": user_input}
        ],
        temperature=0.2,
        response_format={"type": "json_object"} if json_mode else {"type": "text"}
    )
    return res.choices[0].message.content

raw_idea = "A mobile app connecting local bakers with people who want custom birthday cakes."


### Execute the Multi-Agent Pipeline

In [ ]:
# AGENT 1: Market Researcher
researcher_prompt = """You are a Market Researcher.
INPUT: A raw product idea.
OUTPUT FORMAT: Return exactly 3 bullet points:
- Target Audience: [who it is for]
- Market Gap: [what is missing currently]
- Core Problem: [the primary pain point]
Do not add any conversational text."""
research = call_agent(researcher_prompt, f"RAW IDEA:\n{raw_idea}")
print("=== AGENT 1 (RESEARCH) ==\n", research)

# AGENT 2: Tech PM (Parallel Lens A)
tech_pm_prompt = """You are a Technical Product Manager.
INPUT: Market Research data.
OUTPUT FORMAT: Return exactly 3 core MVP features that can be built in 4 weeks. Format as:
1. [Feature Name]: [Brief description]
2. [Feature Name]: [Brief description]
3. [Feature Name]: [Brief description]
Ruthlessly cut feature bloat."""
tech_scope = call_agent(tech_pm_prompt, f"RESEARCH:\n{research}")
print("\n=== AGENT 2 (TECH PM) ==\n", tech_scope)

# AGENT 3: Brand Strategist (Parallel Lens B)
brand_prompt = """You are a Brand Strategist.
INPUT: Market Research data.
OUTPUT FORMAT: Return exactly two lines:
Brand Positioning: [2 sentence value proposition]
Brand Tone: [3 keywords, e.g., Playful, Trustworthy, Bold]"""
brand_strategy = call_agent(brand_prompt, f"RESEARCH:\n{research}")
print("\n=== AGENT 3 (BRAND STRATEGIST) ==\n", brand_strategy)

# AGENT 4: Critic (Quality Gate)
critic_prompt = """You are a Quality Gate Critic.
INPUT: Tech PM scope and Brand Strategy.
OUTPUT FORMAT: Return a short paragraph identifying any conflicts between the technical scope and brand promises. If none, state 'No conflicts identified'."""
critique = call_agent(critic_prompt, f"TECH SCOPE:\n{tech_scope}\n\nBRAND STRATEGY:\n{brand_strategy}")
print("\n=== AGENT 4 (CRITIC) ==\n", critique)

# AGENT 5: Reviser / Synthesizer
reviser_prompt = """You are the Lead Synthesizer for the Idiofy product agency.
Take the critique, resolve the conflicts, and output a structured JSON Go-To-Market Launch Brief.
Schema:
{
  "product_name": "string",
  "target_audience": "string",
  "four_week_v1_features": ["list"],
  "brand_positioning": "string",
  "resolved_tradeoffs": "string"
}
"""
final_brief = call_agent(reviser_prompt, f"TECH:\n{tech_scope}\n\nBRAND:\n{brand_strategy}\n\nCRITIQUE:\n{critique}", json_mode=True)
print("\n=== AGENT 5 (FINAL JSON BRIEF) ==\n", json.dumps(json.loads(final_brief), indent=2))
